# Check FOV completeness

Confirms every expected raw image file for this experiment is actually
present and intact -- not just "the file exists", but (for `.zarr` files)
that every one of its chunks was written completely. A write that gets
interrupted partway through one chunk (a network blip, a killed process, a
disk hiccup on the acquisition PC) leaves a file that *looks* present and
even opens fine, but fails the first time anything tries to actually
decompress the damaged chunk -- this has happened before, surfacing much
later as a cryptic `blosc decompression` error deep inside an unrelated
downstream analysis job, long after the round that wrote it is done.

This notebook checks the whole dataset up front instead: for every
already-finished round, and every FOV, it confirms the file exists, has
the expected number of frames, and every chunk's actual size on disk
matches what that chunk's own compression header declares -- without
decompressing any pixel data, so it stays fast even at full dataset scale.
A full serial scan is still too slow for a real experiment (~1000+ FOVs x
a dozen-plus rounds), so this runs as a SLURM array job, one task per FOV
(`MERci.analysis.cli_check_fov_completeness`), the same pattern
`08_measure_tissue_thickness.ipynb` uses for its own heaviest per-FOV
reads.

## 1 — Setup

In [ ]:
import os
import sys
import csv
import json
from pathlib import Path

import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress_display       import ProgressReporter
from MERci.analysis.completeness  import check_one_file, load_completeness_results
from MERci.acquisition.cluster_submit import (
    build_fov_completeness_array_script, submit_sbatch, is_job_active,
)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")


## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote -- chunk-integrity checking only supports .zarr

# Which rounds to check -- None = every round HAL has finished writing so far
# (meta.round_fully_written); set an explicit list (e.g. [1, 2, 3]) to check
# specific rounds instead, including ones still being imaged (any FOV not
# yet written there will simply show up as "missing_file").
ROUND_IDS = None

# Submit the per-FOV checks as a SLURM array job (recommended -- a serial
# scan over a real experiment takes hours) vs. run locally/serially, which
# is only practical for a small dataset or a quick local test.
USE_SLURM_ARRAY        = True
SLURM_ARRAY_CONCURRENCY = 50
SLURM_MEM               = "2gb"
SLURM_TIME              = "00:15:00"

print(f"Sample name    : {SAMPLE_NAME}")
print(f"Positions tag  : {POSITIONS_TAG}")
print(f"Round IDs      : {'auto (fully-written rounds)' if ROUND_IDS is None else ROUND_IDS}")


## 3 — Resolve config, metadata, and which rounds to check

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)

NOTEBOOK_NAME = "check_fov_completeness"
cache_dir  = config.analysis_dir / "cache" / NOTEBOOK_NAME
output_dir = cache_dir / "per_fov"
cache_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

round_ids = sorted(ROUND_IDS) if ROUND_IDS is not None else sorted(
    rid for rid in meta.valid_round_ids() if meta.round_fully_written(rid)
)

print(f"Rounds  : {meta.n_rounds} total, {len(round_ids)} to check -- {round_ids}")
print(f"FOVs    : {meta.n_fovs}")
print(f"Cache   : {cache_dir}")


## 4 — Check completeness (SLURM array, one task per FOV)

Each array task (`cli_check_fov_completeness.py`) checks one FOV's file
for every round in `round_ids` and writes
`fov<fov_id>_completeness.csv` to `output_dir` -- a FOV is skipped on a
re-run only once its cached file already covers every round currently in
`round_ids` (so a FOV re-run here after a new round finishes gets
re-checked, covering the newly-added round too).

In [ ]:
def fov_cache_path(fov_id):
    return output_dir / f"fov{fov_id:04d}_completeness.csv"

def rounds_already_checked(fov_id):
    p = fov_cache_path(fov_id)
    return set(pd.read_csv(p)["round_id"].unique().tolist()) if p.exists() else set()

all_fov_ids = sorted(meta.fovs)
round_ids_set = set(round_ids)
to_compute = [f for f in all_fov_ids if not round_ids_set <= rounds_already_checked(f)]
print(f"{len(all_fov_ids) - len(to_compute)} / {len(all_fov_ids)} FOV(s) already checked against the "
      f"current {len(round_ids)} round(s); {len(to_compute)} more needed.")

if to_compute and USE_SLURM_ARRAY:
    job_sentinel = cache_dir / "completeness_job.json"
    cached_job = json.loads(job_sentinel.read_text()) if job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute)
            and cached_job.get("round_ids") == round_ids and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / "completeness_manifest.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id"])
            for fov_id in to_compute:
                writer.writerow([fov_id])

        script_path = cache_dir / "completeness.sh"
        build_fov_completeness_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=output_dir,
            round_info_csv=config.round_info_csv, positions_txt=config.positions_txt,
            data_dir=config.data_dir, image_suffix=config.image_suffix, round_ids=round_ids,
            n_pending=len(to_compute), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY, mem=SLURM_MEM, time=SLURM_TIME,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute), "round_ids": round_ids}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute:
    reporter = ProgressReporter(total=len(to_compute), label="Checking FOV completeness (local)")
    for fov_id in reporter.wrap(to_compute):
        rows = []
        for fpath in meta.files_for_fov(fov_id):
            rid = meta.round_id_of_file(fpath)
            if rid not in round_ids_set:
                continue
            row = check_one_file(fpath)
            row["round_id"], row["fov_id"] = rid, fov_id
            rows.append(row)
        pd.DataFrame(rows).to_csv(fov_cache_path(fov_id), index=False)
else:
    print("All FOVs already checked for the current round set.")


## 5 — Results

In [ ]:
results = load_completeness_results(output_dir)
results = results[results["round_id"].isin(round_ids)]
n_checked_fovs = results["fov_id"].nunique()

print(f"Loaded results for {n_checked_fovs} / {len(all_fov_ids)} FOV(s), {len(results)} file(s) total "
      f"across {len(round_ids)} round(s).")
print()
print(results["status"].value_counts().to_string())


Every status other than `ok` (fully intact) or `not_checked` (a non-`.zarr`
file, existence-only) needs a look: `missing_file` (never written),
`missing_chunks`/`truncated_chunks` (written but incomplete/corrupted --
the failure mode this notebook exists to catch), or an `error: ...` status
(a `.zarr` file that isn't the expected v2/blosc format, so it couldn't be
chunk-checked).

In [ ]:
flagged = results[~results["status"].isin(["ok", "not_checked"])].sort_values(["round_id", "fov_id"])
print(f"{len(flagged)} flagged file(s) out of {len(results)} checked.")
flagged
